# SIRA and CoDA Comparison on Colab A100

This notebook compares SIRA and CoDA using three similarly sized 7-8B attack models:

- Llama 3 8B Instruct
- Qwen 2.5 7B Instruct
- Mistral 7B Instruct v0.3

The three models are attack/rewrite models. SIRA and CoDA attack the same 500 shared KGW-watermarked texts generated with OPT-1.3B. All three use bf16 for a controlled comparison.



In [ ]:
# Check that Colab assigned a supported GPU.
import subprocess

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()

print("GPU:", gpu_name)
if "A100" not in gpu_name and "L4" not in gpu_name:
    raise RuntimeError("An A100 or L4 GPU runtime is required.")


GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# Experiment settings for the full 500-sample comparison.
REPO_URL = "https://github.com/hanifnoerr/Self-information-Rewrite-Attack.git"
BRANCH = "codex/browser-colab-l4"
REPO_DIR = "/content/Self-information-Rewrite-Attack"
OUTPUT_ROOT = "/content/drive/MyDrive/sira_3model_500_outputs"

ALGORITHM = "KGW"
SAMPLES = 500
BATCH_SIZE = 32
RESET_OUTPUTS = False
MATRIX_CONFIG = f"{REPO_DIR}/config/model_matrix_l4.json"

print(f"Algorithm: {ALGORITHM}")
print(f"Samples: {SAMPLES}")
print(f"Batch size: {BATCH_SIZE}")
print("Models: Llama 8B, Qwen 2.5 7B, and Mistral 7B; all bf16")


Algorithm: KGW
Samples: 500
Batch size: 32
Models: Llama 8B, Qwen 2.5 7B, and Mistral 7B; all bf16


In [ ]:
# Clone the adapted repository into /content.
from pathlib import Path
import shutil
import subprocess

repo_path = Path(REPO_DIR)
if repo_path.exists():
    shutil.rmtree(repo_path)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR],
    check=True,
)
print("Cloned repository to:", REPO_DIR)


Cloned repository to: /content/Self-information-Rewrite-Attack


In [ ]:
# Install requirements.
subprocess.run(
    ["pip", "install", "-r", f"{REPO_DIR}/requirements.txt"],
    check=True,
)
subprocess.run(
    ["pip", "install", "--upgrade", "transformers", "accelerate"],
    check=True,
)
print("Requirements installed.")


Requirements installed.


In [ ]:
# Log in once. The gated Llama checkpoint requires accepted Hugging Face access.
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Using the HF_TOKEN Colab Secret.")
else:
    print("No HF_TOKEN found. Gated model runs will be marked skipped.")


Using the HF_TOKEN Colab Secret.


In [ ]:
# Mount Drive first so the long run can resume after a Colab disconnect.
import os
from google.colab import drive

drive.mount("/content/drive")

if RESET_OUTPUTS and Path(OUTPUT_ROOT).exists():
    shutil.rmtree(OUTPUT_ROOT)
    print("Removed old outputs so every model uses the same fresh data.")

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

models_config_path = f"{OUTPUT_ROOT}/model_runs.json"
coda_models_config_path = f"{OUTPUT_ROOT}/coda_model_runs.json"
env = os.environ.copy()
env["PYTHONPATH"] = REPO_DIR
env["ALGORITHM"] = ALGORITHM
env["SAMPLES"] = str(SAMPLES)
env["BATCH_SIZE"] = str(BATCH_SIZE)
env["OUTPUT_ROOT"] = OUTPUT_ROOT


Mounted at /content/drive


In [ ]:
# Run SIRA with all three attack models.
subprocess.run(
    [
        "python", "scripts/run_model_matrix_l4.py",
        "--config_path", MATRIX_CONFIG,
        "--repo_dir", REPO_DIR,
        "--output_root", OUTPUT_ROOT,
        "--algorithm", ALGORITHM,
        "--samples", str(SAMPLES),
        "--batch_size", str(BATCH_SIZE),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

# Run CoDA with the same three attack models.
subprocess.run(
    [
        "python", "scripts/run_coda_matrix_l4.py",
        "--config_path", MATRIX_CONFIG,
        "--repo_dir", REPO_DIR,
        "--input_path", f"{OUTPUT_ROOT}/watermarked/{ALGORITHM}_response.json",
        "--output_root", OUTPUT_ROOT,
        "--threshold", "30",
        "--samples", str(SAMPLES),
        "--batch_size", str(BATCH_SIZE),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)
print("CoDA model status:", coda_models_config_path)

# Update the environment record with both SIRA and CoDA model statuses.
subprocess.run(
    ["python", "scripts/write_environment.py", "--output_path", f"{OUTPUT_ROOT}/environment.json", "--models_config", models_config_path, "--coda_models_config", coda_models_config_path, "--batch_size", str(BATCH_SIZE)],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


CoDA model status: /content/drive/MyDrive/sira_3model_500_outputs/coda_model_runs.json


CompletedProcess(args=['python', 'scripts/write_environment.py', '--output_path', '/content/drive/MyDrive/sira_3model_500_outputs/environment.json', '--models_config', '/content/drive/MyDrive/sira_3model_500_outputs/model_runs.json', '--coda_models_config', '/content/drive/MyDrive/sira_3model_500_outputs/coda_model_runs.json', '--batch_size', '32'], returncode=0)

In [ ]:
# Evaluate attack success and semantic preservation against the paper values.
subprocess.run(
    [
        "python", "scripts/evaluate_sira_transfer.py",
        "--generation_model", "facebook/opt-1.3b",
        "--algorithm", ALGORITHM,
        "--watermarked_input", f"{OUTPUT_ROOT}/watermarked/{ALGORITHM}_response.json",
        "--models_config", models_config_path,
        "--coda_models_config", coda_models_config_path,
        "--output_root", OUTPUT_ROOT,
        "--dtype", "bf16",
        "--max_samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

subprocess.run(
    [
        "python", "scripts/compare_transfer_results.py",
        "--output_root", OUTPUT_ROOT,
        "--algorithm", ALGORITHM,
        "--samples", str(SAMPLES),
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)


CompletedProcess(args=['python', 'scripts/compare_transfer_results.py', '--output_root', '/content/drive/MyDrive/sira_3model_500_outputs', '--algorithm', 'KGW', '--samples', '500'], returncode=0)

In [ ]:
# Display the final report and paper-style comparison dataframe.
import pandas as pd
from IPython.display import display

report_path = Path(OUTPUT_ROOT) / "final_report.md"
print(report_path.read_text(encoding="utf-8"))

comparison_dataframe = pd.read_csv(f"{OUTPUT_ROOT}/results/paper_style_comparison.csv")
comparison_columns = [
    "attack_method", "method", "paper_method", "model_family", "size_tier", "parameter_size", "batch_size", "run_status",
    "paper_attack_success_rate", "reproduced_attack_success_rate", "difference", "coda_minus_sira_asr",
    "semantic_similarity", "average_anchor_count", "average_anchor_rate", "note",
]
display(comparison_dataframe[comparison_columns])

print("\nSaved output files:")
for path in sorted(Path(OUTPUT_ROOT).rglob("*")):
    if path.is_file():
        print(path)


# SIRA Cross-Model Transfer Report

## Goal

Test whether the same SIRA attack pipeline transfers across different LLM families and practical size tiers. This can provide evidence of transferability, but it cannot prove SIRA works on every LLM.

## Experiment

- Watermark algorithm: KGW
- Samples: 500
- Shared watermarked data: OPT-1.3B generation on the same C4 subset
- SIRA threshold: 30
- Compared methods: SIRA and CoDA
- Attack models: configured cross-model transfer matrix
- Strong-transfer criterion: ASR >= 0.80 and semantic similarity >= 0.75

Model size and quantization are recorded because they are possible confounders. In particular, a 4-bit result should not be treated as a precision-controlled comparison with a bf16 result.

The configured tiers are practical single-GPU tiers rather than perfectly parameter-matched controls.

## Results

| attack method | configuration | family | tier | size | quantization | batch | status | ASR | semantic similarity | average watermark sco

,attack_method,method,paper_method,model_family,size_tier,parameter_size,batch_size,run_status,paper_attack_success_rate,reproduced_attack_success_rate,difference,coda_minus_sira_asr,semantic_similarity,average_anchor_count,average_anchor_rate,note
0,SIRA,Llama 3 8B Instruct,SIRA-Small,Llama,7-8B,8B,32,completed,1.0,0.988,-0.012,-0.472,0.812999,NaN,NaN,Released-code checkpoint for paper SIRA-Small;...
1,SIRA,Qwen 2.5 7B Instruct,NaN,Qwen,7-8B,7.61B,32,completed,NaN,0.984,NaN,-0.670,0.866957,NaN,NaN,Cross-model SIRA transfer test.
2,SIRA,Mistral 7B Instruct v0.3,NaN,Mistral,7-8B,7B,32,completed,NaN,0.952,NaN,-0.488,0.856315,NaN,NaN,Cross-model SIRA transfer test.
3,CoDA,CoDA - Llama 3 8B Instruct,NaN,Llama,7-8B,8B,32,completed,NaN,0.516,NaN,-0.472,0.816004,84.854,0.344623,Proposed CoDA method. It changes low-informati...
4,CoDA,CoDA - Qwen 2.5 7B Instruct,NaN,Qwen,7-8B,7.61B,32,completed,NaN,0.314,NaN,-0.670,0.909808,86.026,0.343393,Proposed CoDA method. It changes low-informati...
5,CoDA,CoDA - Mistral 7B Instruct v0.3,NaN,Mistral,7-8B,7B,32,completed,NaN,0.464,NaN,-0.488,0.898686,95.128,0.335979,Proposed CoDA method. It changes low-informati...



Saved output files:
/content/drive/MyDrive/sira_3model_500_outputs/coda_model_runs.json
/content/drive/MyDrive/sira_3model_500_outputs/coda_models/llama_3_8b/coda_attack.jsonl
/content/drive/MyDrive/sira_3model_500_outputs/coda_models/mistral_7b_v0_3/coda_attack.jsonl
/content/drive/MyDrive/sira_3model_500_outputs/coda_models/qwen_2_5_7b/coda_attack.jsonl
/content/drive/MyDrive/sira_3model_500_outputs/environment.json
/content/drive/MyDrive/sira_3model_500_outputs/final_report.md
/content/drive/MyDrive/sira_3model_500_outputs/logs/coda_llama_3_8b.log
/content/drive/MyDrive/sira_3model_500_outputs/logs/coda_mistral_7b_v0_3.log
/content/drive/MyDrive/sira_3model_500_outputs/logs/coda_qwen_2_5_7b.log
/content/drive/MyDrive/sira_3model_500_outputs/logs/sira_llama_3_8b.log
/content/drive/MyDrive/sira_3model_500_outputs/logs/sira_mistral_7b_v0_3.log
/content/drive/MyDrive/sira_3model_500_outputs/logs/sira_qwen_2_5_7b.log
/content/drive/MyDrive/sira_3model_500_outputs/model_runs.json
/conten

## Export Exact KGW Token Colors

SIRA masks model tokens rather than complete words, so the intermediate text can contain word fragments and punctuation. This cell exports the exact KGW green-list and red-list assignments on the same A100 runtime, then builds the result dashboard.


In [ ]:
# Export exact KGW token classes for representative samples.
VISUAL_SAMPLE_IDS = ["0", "100", "200", "300", "400"]

subprocess.run(
    [
        "python", "scripts/export_kgw_token_colors.py",
        "--output_root", OUTPUT_ROOT,
        "--sample_ids", *VISUAL_SAMPLE_IDS,
        "--required_gpu_substring", "A100",
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

# Build the interactive dashboard and metrics figure.
subprocess.run(
    [
        "python", "scripts/visualize_sira_results.py",
        "--output_root", OUTPUT_ROOT,
        "--sample_ids", *VISUAL_SAMPLE_IDS,
    ],
    cwd=REPO_DIR,
    env=env,
    check=True,
)

from IPython.display import Image, display

visualization_dir = Path(OUTPUT_ROOT) / "visualization"
display(Image(filename=str(visualization_dir / "metrics_overview.png")))
print("Interactive dashboard:", visualization_dir / "index.html")
print("Exact KGW token export:", visualization_dir / "kgw_token_colors.json")
